# Background

This script does the following:  
- Drops any columns where all data is NA
- Drops fields that aren't in Ameriflux
- Review final list of fields
- Turn CO2 and H2O variables to nan based on signal strength flags
- Convert NA values to -9999


- create entyr for the processing table in the atabgase
- name output file correctly!

# Setup

In [ ]:
station = "US-UTJ"

data_interval = 'HH'

date_range = '202406051200_202510091200'
parquet_path = r'M:\Shared drives\UGS_Flux\Data_Processing\final_database_tables' 
file_name = f'{station}_{date_range}_qc.parquet'
micromet_path = r"C:/Users/dmenuz/Documents/scripts/MicroMet/src/"
amflux_dat = r'M:\Shared drives\UGS_Flux\Data_Processing\site_specific_data_review\flux-met_processing_variables_20250818.csv'

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.append(micromet_path)
from micromet import validate
from micromet import timestamps

In [ ]:
file_path = Path (parquet_path, 'qc', file_name)
qc_dat = pd.read_parquet(file_path)
amflux = pd.read_csv(amflux_dat)

# Cleanup


## Drop Data Based on Signal Strength

In [ ]:
# Drop based on H2O sig strenght
mask = qc_dat.H2O_SIG_STRGTH_MIN_1_1_1<0.8
print(f'{mask.sum()} records dropped due to signal strenght')
h2o_drop_vars = ['H2O_1_1_1', 'H2O_SIGMA_1_1_1', 'LE_1_1_1', 'RH_1_1_1', 'RH_1_2_1', 'VPD_1_1_1', 'ET_1_1_1']
qc_dat.loc[mask, h2o_drop_vars] = np.nan

In [ ]:
# Drop based on CO2 sig strength
mask = qc_dat.CO2_SIG_STRGTH_MIN_1_1_1<0.8
print(f'{mask.sum()} records dropped due to signal strength')
co2_drop_vars = ['CO2_1_1_1', 'CO2_SIGMA_1_1_1', 'FC_1_1_1']
qc_dat.loc[mask, co2_drop_vars] = np.nan

## Drop Irrelevant Columns

You MUST manually review this to make sure you have the columns correct

In [ ]:
# drop any columns that are all NA values
all_null = qc_dat.isnull().all()
if all_null.any():
    print("DROPPING COLUMNS WITH ALL NULL VALUES:")
    print(qc_dat.columns[all_null])
    qc_dat = qc_dat.drop(columns=qc_dat.columns[all_null])
else:
    print("NO COLUMNS WITH ALL NULL VALUES TO DROP.")

In [ ]:
# Obtain initial list of columns to drop based on Ameriflux naming comparison
results = validate.compare_names_to_ameriflux(qc_dat, amflux)

drop_col_mask = results.is_in_amflux==False
drop_results = results[drop_col_mask]

cols_to_drop = drop_results.all_columns

keep_col = ['PBLH_F'] # add any other ones that you want to keep here
cols_to_drop = cols_to_drop[~cols_to_drop.isin(keep_col)]

drop_cols = ['WD_1_1_1_FLAG', 'ZL_1_1_1', 'FETCH_MAX', 'FETCH_90']
cols_to_drop = cols_to_drop._append(pd.Series(drop_cols)).drop_duplicates()

In [ ]:
# review columns to drop based on Ameriflux naming
print("COLUMNS TO DROP:")
print('\n')
for col in cols_to_drop.sort_values().unique():
    print(col)

qc_dat = qc_dat.drop(columns=cols_to_drop)

In [ ]:
# review remaining columns that will go to Ameriflux
for col in qc_dat.columns.sort_values():
    print(col)

## Turn nan to -9999

In [ ]:
qc_dat = qc_dat.fillna(-9999)

## Recalculate timestamp

In [ ]:
qc_dat = timestamps.add_ameriflux_timestamps(qc_dat, interval_minutes=30)
assert (qc_dat.TIMESTAMP_START.isna().sum() == 0) & (qc_dat.TIMESTAMP_END.isna().sum() == 0), "TIMESTAMP_START or TIMESTAMP_END contains NA values after processing."

In [ ]:
# manually review timestamps
qc_dat[['TIMESTAMP_START','TIMESTAMP_END']].head(3)

# Compare with existing Ameriflux data

Compare with data that are already in Ameriflux to make sure all of the data will be ovewrritten  
You can see the log dates here:  
 https://docs.google.com/spreadsheets/d/17GPIBjg_GpkAx4uAxOBcFWPDvIfM-k1S3TfjmcWIocw/edit?gid=0#gid=0

In [ ]:
# get this from the log above for your station!
initial_submission_date = 202406261230
timestamp_start = qc_dat.TIMESTAMP_START.min()
timestamp_end = qc_dat.TIMESTAMP_END.max()

In [ ]:
print(f'Initial submission start date: {initial_submission_date}')
print(f'Current submission start date: {timestamp_start}')
assert initial_submission_date>=int(timestamp_start), "Initial submission date is before the start of the new data"

# Export Data

In [ ]:
file_name = f"{station}_{data_interval}_{timestamp_start}_{timestamp_end}.csv"
file_path = Path(f'{parquet_path}', "ameriflux", f'{file_name}')
qc_dat.to_csv(file_path)
from datetime import datetime
print(f'Data exported {datetime.now()}')
print(f'Exported data to {file_path}')